In [1]:
import uproot
import numpy as np
import glob
import json
import ROOT

In [2]:
base_dir  = "../Dataset_ver2/Data/reduce_root"

# Collect all ROOT files recursively (A–L, run folders)
root_files = glob.glob(f"{base_dir}/*/run*/root_*_*.root", recursive=True)
print(f"Found {len(root_files)} ROOT files")

Found 44688 ROOT files


In [3]:
with open("ATLAS.json", "r") as f:
    metadata = json.load(f)

def get_root_links(run):
    links = []
    for meta_run in metadata["metadata"]["_file_indices"]:
        if meta_run["key"].split("_")[3][2:] == run:
            for root_file in meta_run["files"]:
                links.append(root_file["uri"])
    return links

run_to_links = {}
for root_file in root_files:
    run = root_file.split("/")[-2].replace("run", "")

    if run not in run_to_links:
        run_to_links[run] = get_root_links(run) 

In [4]:
branch_name = "PrimaryVerticesAuxDyn.z"
chain = ROOT.TChain("CollectionTree")

for link in run_to_links["306269"]:
    chain.Add(link)
    # print(f"Added {len(links)} files for run {run}, total entries: {chain.GetEntries()}")
# print(f"Total entries in chain: {chain.GetEntries()}")
print("Creating RDataFrame...")
df = ROOT.RDataFrame(chain)
print("Creating snapshot...")
df.Snapshot("CollectionTree", f"zVertex.root", branch_name)

Creating RDataFrame...
Creating snapshot...


<cppyy.gbl.ROOT.RDF.RResultPtr<ROOT::RDF::RInterface<ROOT::Detail::RDF::RLoopManager,void> > object at 0x58c9f60fa640>

Plugin No such file or directory loading sec.protocol libXrdSeckrb5-5.so
Warning in <TClass::Init>: no dictionary for class DataHeader_p6 is available
Warning in <TClass::Init>: no dictionary for class DataHeaderForm_p6 is available
Warning in <TClass::Init>: no dictionary for class xAOD::AuxContainerBase is available
Warning in <TClass::Init>: no dictionary for class ElementLinkBase is available
Warning in <TClass::Init>: no dictionary for class xAOD::AuxInfoBase is available
Warning in <TClass::Init>: no dictionary for class xAOD::EventInfo_v1 is available
Warning in <TClass::Init>: no dictionary for class xAOD::TrigConfKeys_v1 is available
Warning in <TClass::Init>: no dictionary for class xAOD::TrigDecisionAuxInfo_v1 is available
Warning in <TClass::Init>: no dictionary for class xAOD::TrigDecision_v1 is available
Warning in <TClass::Init>: no dictionary for class xAOD::EventShape_v1 is available
Warning in <TClass::Init>: no dictionary for class ElementLink<DataVector<xAOD::Vertex

In [5]:
# # root_files[10]



# z_values = []

# for path in root_files:
#     with uproot.open(path) as file:
#         # Inspect tree name if unsure
#         tree = file[file.keys()[0]]
        
#         # Check available branches once
#         # print(tree.keys())

#         # Load only the branch you want
#         if branch_name in tree.keys():
#             z = tree[branch_name].array(library="np")
#             z_values.append(z)
#         else:
#             print(f"Branch {branch_name} not found in {path}")

# # Flatten if needed
# all_z = np.concatenate(z_values)
# print("Total vertices:", len(all_z))
# print("Mean z:", np.mean(all_z))